# End-to-end test against Azure App Service (v2)

Tests the v5.2 pipeline through the deployed Flask API.

**Defensive error handling:**
- Every HTTP call checks status code AND content-type before parsing
- Non-JSON responses show the actual body so you can see App Service
  error pages, 502s, etc.
- Health check fails fast if v5.2 isn't deployed

**Polling design:**
- Uses lightweight `/status/<load_id>` for polling (no row data)
- Refreshes a single line via `clear_output(wait=True)`
- Polls every 30 seconds
- Maximum wait: 2 hours
- Set `RESUME_LOAD_ID` in cell 0 to resume polling an existing run

## 0. Config

In [ ]:
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path

import requests
from IPython.display import clear_output

# ----- Server -----
BASE_URL = 'https://desrapidev-fqaphqeaeuc7hafh.westcentralus-01.azurewebsites.net'

# ----- Input -----
PAYLOAD_PATH = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_payload.json')

# ----- Version verification -----
EXPECTED_BUILD_VERSION = '2026-05-14-fast-pipeline-v5.2'

# ----- Polling -----
POLL_INTERVAL = 30
MAX_WAIT      = 7200
POST_TIMEOUT  = 180

# ----- Resume mode -----
RESUME_LOAD_ID = None   # set to e.g. '210' to resume an existing run

print(f'Server:           {BASE_URL}')
print(f'Payload:          {PAYLOAD_PATH.name}')
print(f'Expected version: {EXPECTED_BUILD_VERSION}')
print(f'Resume load_id:   {RESUME_LOAD_ID or "(starting fresh)"}')


## 1. Safe HTTP helper

Returns parsed JSON, or raises a clear error explaining what came back
instead. Use for every server call.

In [ ]:
def safe_get_json(url, timeout=30, allow_status=(200, 202)):
    '''GET a URL and return parsed JSON. On non-JSON or unexpected status,
    raise with the actual response body so the failure mode is visible.'''
    try:
        r = requests.get(url, timeout=timeout)
    except requests.exceptions.ConnectionError as e:
        raise RuntimeError(f'Connection error - is the server reachable?\n  URL: {url}\n  Error: {e}')
    except requests.exceptions.Timeout:
        raise RuntimeError(f'Request timed out after {timeout}s\n  URL: {url}')

    if r.status_code not in allow_status:
        # Show the actual body so the user can see App Service error pages, auth challenges, etc.
        body_preview = r.text[:800] if r.text else '(empty body)'
        raise RuntimeError(
            f'HTTP {r.status_code} (expected one of {allow_status})\n'
            f'  URL: {url}\n'
            f'  Body preview:\n{body_preview}'
        )

    content_type = r.headers.get('Content-Type', '')
    if 'json' not in content_type.lower():
        body_preview = r.text[:800] if r.text else '(empty body)'
        raise RuntimeError(
            f'Expected JSON response but got Content-Type={content_type!r}\n'
            f'  URL: {url}\n'
            f'  Body preview:\n{body_preview}'
        )

    try:
        return r.json(), r.status_code
    except json.JSONDecodeError as e:
        body_preview = r.text[:800] if r.text else '(empty body)'
        raise RuntimeError(
            f'Response claimed Content-Type=json but failed to parse: {e}\n'
            f'  URL: {url}\n'
            f'  Body preview:\n{body_preview}'
        )


def safe_post_json(url, payload, timeout=180, allow_status=(202,)):
    '''POST JSON and return parsed JSON response. Same error-handling discipline as safe_get_json.'''
    try:
        r = requests.post(url, json=payload, headers={'Content-Type': 'application/json'}, timeout=timeout)
    except requests.exceptions.ConnectionError as e:
        raise RuntimeError(f'Connection error\n  URL: {url}\n  Error: {e}')
    except requests.exceptions.Timeout:
        raise RuntimeError(f'POST timed out after {timeout}s\n  URL: {url}')

    if r.status_code not in allow_status:
        body_preview = r.text[:800] if r.text else '(empty body)'
        raise RuntimeError(
            f'HTTP {r.status_code} (expected one of {allow_status})\n'
            f'  URL: {url}\n'
            f'  Body preview:\n{body_preview}'
        )

    content_type = r.headers.get('Content-Type', '')
    if 'json' not in content_type.lower():
        body_preview = r.text[:800] if r.text else '(empty body)'
        raise RuntimeError(
            f'Expected JSON response but got Content-Type={content_type!r}\n'
            f'  URL: {url}\n'
            f'  Body preview:\n{body_preview}'
        )

    return r.json(), r.status_code

print('HTTP helpers ready.')


## 2. Health check - verify server is on v5.2

If this fails with a non-200 or non-JSON response, the error message will
show you EXACTLY what came back from the server.

In [ ]:
health, status_code = safe_get_json(f'{BASE_URL}/', timeout=30)
print(f'HTTP {status_code}')
print(json.dumps(health, indent=2))

server_ver = health.get('build_benefits_ver')
if server_ver != EXPECTED_BUILD_VERSION:
    raise SystemExit(
        f'\nSTOP - server reports build {server_ver!r}, expected {EXPECTED_BUILD_VERSION!r}.\n'
        f'  Redeploy build_benefits.py + checkpoint_runner.py and restart App Service.'
    )

print(f'\nOK - server is on v5.2 (fast-pipeline with skip-empty + concurrent plans)')


## 3. Load and sanitize the payload

Reads local JSON, strips NaN/Inf (JSON can't carry them), and reports
what's in the payload.

In [ ]:
def sanitize(obj):
    '''Replace NaN/Inf floats with None so json.dumps works.'''
    if isinstance(obj, dict):
        return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    return obj

raw = json.loads(PAYLOAD_PATH.read_text(encoding='utf-8'))
rows = raw['pbp'] if isinstance(raw, dict) and 'pbp' in raw else raw
clean = sanitize(rows)

input_plans = {r['FileName'].strip() for r in clean if r.get('FileName')}
input_load_ids = {str(r['LoadID']).strip() for r in clean if r.get('LoadID')}

print(f'PBP rows:        {len(clean):,}')
print(f'Distinct plans:  {len(input_plans)}')
print(f'LoadIDs:         {sorted(input_load_ids)}')

if len(input_load_ids) != 1:
    raise SystemExit(f'STOP - payload must contain exactly one LoadID. Found: {input_load_ids}')

print(f'\nFirst 5 plan filenames:')
for fn in sorted(input_plans)[:5]:
    print(f'  - {fn}')


## 4. POST the payload (skipped if RESUME_LOAD_ID is set)

Returns a `load_id` in a few seconds. Server processes in the background.

In [ ]:
if RESUME_LOAD_ID:
    load_id = RESUME_LOAD_ID
    post_time = 0.0
    print(f'Resuming poll of load_id={load_id} (skipping POST)')
else:
    t_post = time.monotonic()
    resp, status_code = safe_post_json(f'{BASE_URL}/save', clean, timeout=POST_TIMEOUT)
    post_time = time.monotonic() - t_post

    print(f'POST: HTTP {status_code} in {post_time:.1f}s')
    print(json.dumps(resp, indent=2))

    load_id = resp.get('load_id')
    if not load_id:
        raise SystemExit(f'STOP - server response had no load_id: {resp}')


## 5. Poll /status until complete

Lightweight - returns plan counts only, not the row data. Refreshes a
single output line every 30s using `clear_output(wait=True)`.

In [ ]:
t_proc = time.monotonic()
elapsed = 0
final_status = None
last_n_done = -1
stall_count = 0

print(f'Polling /status/{load_id} every {POLL_INTERVAL}s (max wait: {MAX_WAIT}s)\n')

while elapsed < MAX_WAIT:
    try:
        s, code = safe_get_json(f'{BASE_URL}/status/{load_id}', timeout=30, allow_status=(200, 404))

        if code == 404:
            clear_output(wait=True)
            print(f'[{datetime.now().strftime("%H:%M:%S")}] No status blob yet for {load_id}')
            print(f'  Server may still be initializing. Elapsed: {elapsed}s')
            time.sleep(POLL_INTERVAL)
            elapsed += POLL_INTERVAL
            continue

        n_done = s.get('n_plans_done', 0)
        n_total = s.get('n_plans_total', '?')
        n_failed = s.get('n_plans_failed', 0)
        status = s.get('status')

        if n_done == last_n_done:
            stall_count += 1
        else:
            stall_count = 0
            last_n_done = n_done

        rate_info = ''
        if isinstance(n_done, int) and isinstance(n_total, int) and n_done > 0:
            avg_per_plan = elapsed / n_done if n_done else 0
            remaining_plans = n_total - n_done
            est_remaining = remaining_plans * avg_per_plan
            rate_info = f'\n  avg/plan: {avg_per_plan:.1f}s | ETA: {est_remaining:.0f}s ({est_remaining/60:.1f} min)'

        clear_output(wait=True)
        print(f'[{datetime.now().strftime("%H:%M:%S")}]  load_id={load_id}')
        print(f'  status:       {status}')
        print(f'  plans done:   {n_done}/{n_total}')
        print(f'  plans failed: {n_failed}')
        print(f'  updated_at:   {s.get("updated_at")}')
        print(f'  elapsed:      {elapsed}s ({elapsed/60:.1f} min){rate_info}')

        if stall_count >= 10:
            print(f'\n  WARNING: n_plans_done unchanged for {stall_count * POLL_INTERVAL}s')
            print(f'  Worker may have died. Check App Service log stream.')

        if status in ('success', 'partial', 'error'):
            final_status = s
            break

    except RuntimeError as e:
        # safe_get_json raised - server hiccup, try again
        print(f'[{elapsed}s] poll error: {e}')

    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL

proc_time = time.monotonic() - t_proc

if not final_status:
    raise SystemExit(
        f'\nTimed out after {MAX_WAIT}s without seeing terminal status.\n'
        f'Re-run this notebook with RESUME_LOAD_ID = {load_id!r} to resume polling.'
    )

print(f'\n--- FINAL STATUS ---')
print(f'  status:        {final_status.get("status")}')
print(f'  plans done:    {final_status.get("n_plans_done")}/{final_status.get("n_plans_total")}')
print(f'  plans failed:  {final_status.get("n_plans_failed")}')
print(f'  processing:    {proc_time:.1f}s ({proc_time/60:.1f} min)')
print(f'  build:         {final_status.get("build_version")}')


## 6. Fetch full results

One big GET to pull the combined output. Can be several MB - allow time.

In [ ]:
print(f'Fetching /results/{load_id} ...')
t_fetch = time.monotonic()
result_data, _ = safe_get_json(f'{BASE_URL}/results/{load_id}', timeout=180, allow_status=(200, 202))
fetch_time = time.monotonic() - t_fetch

output_rows = result_data.get('results', [])

print(f'Fetched in {fetch_time:.1f}s')
print(f'Output:')
print(f'  result_count:  {result_data.get("result_count")}')
print(f'  plan_count:    {result_data.get("plan_count")}')
print(f'  output_blob:   {result_data.get("output_blob")}')

total_time = post_time + proc_time + fetch_time
print(f'\nWall time breakdown:')
print(f'  POST:        {post_time:>7.1f}s')
print(f'  Processing:  {proc_time:>7.1f}s')
print(f'  Fetch:       {fetch_time:>7.1f}s')
print(f'  TOTAL:       {total_time:>7.1f}s ({total_time/60:.1f} min)')


## 7. Sanity checks on the output

In [ ]:
import pandas as pd

if not output_rows:
    print('No rows in output - cannot run sanity checks')
else:
    df = pd.DataFrame(output_rows)
    print(f'Total rows:           {len(df):,}')
    print(f'Distinct plans:       {df["planid"].nunique()}')
    print(f'Distinct benefit IDs: {df["benefitid"].nunique()}')

    output_plans = set(df['planid'].unique())
    print(f'\nOutput planids match input plans:')
    for fn in sorted(input_plans)[:5]:
        # Loose match - planid format is CARRIER_contract_plan_segment
        parts = fn.split('_', 1)
        if len(parts) == 2:
            contract = parts[1].split('-')[0]
            matching = [p for p in output_plans if contract in p]
            print(f'  {fn} -> matched {len(matching)} planid(s)')


### Rows per plan

In [ ]:
if output_rows:
    rows_per_plan = df.groupby('planid').size().sort_values(ascending=False)
    print(f'Rows per plan ({len(rows_per_plan)} plans):')
    print(f'  Top 15:')
    print(rows_per_plan.head(15).to_string())
    print()
    print(f'  Min:    {rows_per_plan.min()}')
    print(f'  Max:    {rows_per_plan.max()}')
    print(f'  Median: {rows_per_plan.median():.0f}')
    print(f'  Mean:   {rows_per_plan.mean():.1f}')


### Rows per benefit ID

In [ ]:
if output_rows:
    df['benefitid'] = df['benefitid'].astype(str)
    rows_per_bid = df.groupby('benefitid').size().sort_values(ascending=False)
    plans_per_bid = df.groupby('benefitid')['planid'].nunique()

    summary = pd.DataFrame({
        'total_rows':   rows_per_bid,
        'plans_w_data': plans_per_bid,
        'pct_plans':    (100 * plans_per_bid / df['planid'].nunique()).round(0).astype(str) + '%',
    }).sort_values('total_rows', ascending=False)
    print(summary.to_string())


## 8. Per-plan timing - measure parallel speedup

In [ ]:
plans_status = final_status.get('plans', {})
timing_rows = []
for fn, info in plans_status.items():
    if info.get('status') == 'done':
        timing_rows.append({
            'filename':  fn,
            'rows':      info.get('rows', 0),
            'elapsed_s': info.get('elapsed_s', 0),
        })

if timing_rows:
    timing_df = pd.DataFrame(timing_rows).sort_values('elapsed_s', ascending=False)
    print(f'Slowest 15 plans:')
    print(timing_df.head(15).to_string(index=False))
    print()
    print(f'Per-plan stats:')
    print(f'  fastest: {timing_df["elapsed_s"].min():.1f}s')
    print(f'  slowest: {timing_df["elapsed_s"].max():.1f}s')
    print(f'  median:  {timing_df["elapsed_s"].median():.1f}s')
    print(f'  mean:    {timing_df["elapsed_s"].mean():.1f}s')

    sum_per_plan = timing_df['elapsed_s'].sum()
    speedup = sum_per_plan / proc_time if proc_time > 0 else 0
    print()
    print(f'  Sum of per-plan time: {sum_per_plan:.1f}s ({sum_per_plan/60:.1f} min)')
    print(f'  Wall time:            {proc_time:.1f}s ({proc_time/60:.1f} min)')
    print(f'  Parallel speedup:     {speedup:.2f}x')
    if speedup < 2:
        print(f'  NOTE: speedup is low - check CONCURRENT_PLANS env var and Azure OpenAI TPM')

failed_plans = [(fn, info) for fn, info in plans_status.items() if info.get('status') == 'failed']
if failed_plans:
    print(f'\nFailed plans: {len(failed_plans)}')
    for fn, info in failed_plans[:10]:
        print(f'  {fn}: {info.get("error", "")[:120]}')


## 9. Save output locally

In [ ]:
local_out = Path.cwd() / f'output_benefits_{load_id}.json'
local_out.write_text(json.dumps(output_rows, indent=2), encoding='utf-8')
print(f'Saved {len(output_rows):,} rows to {local_out}')
print(f'File size: {local_out.stat().st_size:,} bytes ({local_out.stat().st_size/1024/1024:.2f} MB)')


## 10. Verdict

In [ ]:
print('=' * 70)
print(f'Run summary: load_id={load_id}')
print('=' * 70)
print(f'Build:                {final_status.get("build_version")}')
print(f'Plans:                {final_status.get("n_plans_done")}/{final_status.get("n_plans_total")} done | '
      f'{final_status.get("n_plans_failed")} failed')
print(f'Output rows:          {len(output_rows):,}')
print(f'Wall time:            {total_time/60:.1f} min')
if timing_rows:
    print(f'Sum per-plan time:    {sum_per_plan/60:.1f} min')
    print(f'Parallel speedup:     {speedup:.2f}x')
print()
if final_status.get('status') == 'success':
    print('STATUS: SUCCESS')
    print(f'  Output: outbound/{result_data.get("output_blob")} (server)')
    print(f'  Local:  {local_out}')
elif final_status.get('status') == 'partial':
    pending = (final_status.get('n_plans_total', 0)
               - final_status.get('n_plans_done', 0)
               - final_status.get('n_plans_failed', 0))
    print(f'STATUS: PARTIAL - {pending} pending, {final_status.get("n_plans_failed")} failed')
    print(f'Action: re-POST same payload to resume, or run with RESUME_LOAD_ID = {load_id!r}')
else:
    print(f'STATUS: {final_status.get("status")}')
